# Baselines and models

This notebook loads the daily 4 PM winter table saved by Notebook 02, checks that the saved labels match the expected version, and prepares the whole-winter splits. It does not train a model. Review the printed feature list for future information before modeling.

In [ ]:
import numpy as np
import pandas as pd

daily_4pm = pd.read_csv("../processed_data/daily_4pm_winter_labels.csv",
                        index_col=0, parse_dates=True)

# Days since 1 November of each winter.
start = pd.DatetimeIndex(pd.to_datetime(
    pd.DataFrame({"year": daily_4pm["season"].astype(int).values,
                  "month": 11, "day": 1})))
daily_4pm["season_day"] = (daily_4pm.index - start).days

# Whole-winter splits. Keep test rows untouched until final evaluation.
train_rows = daily_4pm["season"].isin([2018, 2019, 2021, 2022])
val_rows = daily_4pm["season"].eq(2023)
test_rows = daily_4pm["season"].eq(2024)

# The two outcomes are evaluated separately because their known rows differ.
labels = ["target_48h_301", "target_any_48h_301"]

# Load onset timestamps for later event-level evaluation.
onsets = pd.read_csv("../processed_data/winter_onsets.csv",
                     parse_dates=["onset_time"])["onset_time"]

# Check that the saved table matches the current Notebook 02 labels.
n_exact = int(daily_4pm["target_48h_301"].notna().sum())
n_any = int(daily_4pm["target_any_48h_301"].notna().sum())
assert (n_exact, n_any) == (310, 291), f"Stale file? got {n_exact} and {n_any}"

def rows_for(label, split):
    """Keep split rows with known outcomes and a starting AQI below 301."""
    usable = split & daily_4pm["AQI"].lt(301) & daily_4pm[label].notna()
    return daily_4pm.loc[usable]

# Exclude labels, future measurements, and bookkeeping from numeric inputs.
leaky = [c for c in daily_4pm.columns
         if c.startswith(("target_", "AQI_at_plus", "AQI_change")) or "ahead" in c]
bookkeeping = ["season", "starting_aqi_band", "AQI_source", "AQI_review_status"]
feature_cols = [c for c in daily_4pm.columns
                if c not in leaky + bookkeeping
                and pd.api.types.is_numeric_dtype(daily_4pm[c])]

print("Rows:", daily_4pm.shape[0])
print("Train / val / test rows:", int(train_rows.sum()), int(val_rows.sum()), int(test_rows.sum()))
print("Known onset timestamps:", len(onsets))
print("Feature columns (", len(feature_cols), "):", feature_cols)

Rows: 722
Train / val / test rows: 481 121 120
Known onset timestamps: 56
Feature columns ( 25 ): ['PM2.5 (µg/m³)', 'PM10 (µg/m³)', 'NO (µg/m³)', 'NO2 (µg/m³)', 'NOx (ppb)', 'NH3 (µg/m³)', 'SO2 (µg/m³)', 'CO (mg/m³)', 'Ozone (µg/m³)', 'Benzene (µg/m³)', 'Toluene (µg/m³)', 'AT (°C)', 'RH (%)', 'WS (m/s)', 'WD (deg)', 'RF (mm)', 'TOT-RF (mm)', 'SR (W/mt2)', 'BP (mmHg)', 'VWS (m/s)', 'AQI', 'AQI_24h', 'AQI_48h', 'AQI_72h', 'season_day']
